In [ ]:

PRICE_TRANSFORM_SCALE = 100.0



def compute_cross_feats(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with 4 auxiliary cross-feature columns.

    Columns
    -------
    doy_sin            : 365.25-day annual sinusoidal cycle
    doy_cos            : 365.25-day annual cosine cycle
    sa_spread_live     : arcsinh((SA - NSW) / scale) — SA-NSW interconnector pressure
    region_spike_score : count of QLD/VIC/SA regions with price > 150 (0–3)
    """
    n   = len(df)
    doy = df.index.day_of_year.astype(np.float32)

    doy_sin = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    doy_cos = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

    if "sa_price" in df.columns and "nsw_price" in df.columns:
        sa  = df["sa_price"].fillna(0.0).values.astype(np.float32)
        nsw = df["nsw_price"].fillna(0.0).values.astype(np.float32)
        sa_spread = np.arcsinh((sa - nsw) / PRICE_TRANSFORM_SCALE).clip(-10, 10).astype(np.float32)
    elif "nsw_price_vs_sa_price_spread" in df.columns:
        sa_spread = np.arcsinh(
            df["nsw_price_vs_sa_price_spread"].fillna(0.0).values / PRICE_TRANSFORM_SCALE
        ).clip(-10, 10).astype(np.float32)
    else:
        sa_spread = np.zeros(n, dtype=np.float32)

    score = np.zeros(n, dtype=np.float32)
    for reg_col in ["qld_price", "vic_price", "sa_price"]:
        if reg_col in df.columns:
            score += (df[reg_col].fillna(0.0).values > 150).astype(np.float32)

    return pd.DataFrame({
        "doy_sin":            doy_sin,
        "doy_cos":            doy_cos,
        "sa_spread_live":     sa_spread,
        "region_spike_score": score,
    }, index=df.index)



# Price col (clip threshold source) + naive baseline lags (blend baseline).
_price_lag_cols = [
    f"{os.environ.get("TARGET").lower()}_price",
    f"{os.environ.get("TARGET").lower()}_price_lag_336",
    f"{os.environ.get("TARGET").lower()}_price_lag_48",
]
_price_lag_cols = [c for c in _price_lag_cols if c in df.columns]

auxiliary = pd.concat([cross, df[_price_lag_cols]], axis=1)
    # # train_aux = auxiliary.loc[train_df.index]
    # # valid_aux = auxiliary.loc[validate_df.index]


 # Cross-features pre-computed in 2_Features_build/9_auxiliary_features.ipynb.
    # _CROSS_COLS = ["doy_sin", "doy_cos", "sa_spread_live", "region_spike_score"]
    # cross_tr = train_aux[_CROSS_COLS].values.astype(np.float32)
    # cross_va = valid_aux[_CROSS_COLS].values.astype(np.float32)


# aux_vars = train_aux[f"{os.environ["TARGET"].lower()}_price"].values if f"{os.environ["TARGET"].lower()}_price" in train_aux.columns else (
#     train_tgt[[c for c in train_tgt.columns if c.startswith("h") and c[1:].isdigit()]].values.ravel()
# )

# aux_vars = aux_vars[np.isfinite(aux_vars)]